In [ ]:
import os
import sys
import importlib
import pandas as pd
import json
from pyutil.db_entities import *
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

db_file = 'compiled.db'
eng = create_engine(f'sqlite:///{db_file}')
Base.metadata.create_all(eng)
session = Session(eng)
projects = session.scalars(select(Project)).all()

In [ ]:
DATASOURCE = 'llm'
CONFIDENCE_THRESHOLD = 0.80
DISABLE_DOCUMENTATION_CATEGORY = False

df = {}
if DATASOURCE == 'llm':
    df = pd.read_csv('classified_proposals_llm_15_06.csv')    

if DATASOURCE == 'zeroshot':
    df = pd.read_csv('classified_proposals_zeroshot_15_06.csv')

df.head()

In [ ]:
def safe_json_load(val):
    if not val:
        return []
    try:
        if isinstance(val, list):
            return val

        formatted = val.replace('\'', '\"')
        return json.loads(formatted)
    except (json.JSONDecodeError, TypeError):
        return []

#filter out any rows where the score is [0]
df = df[~df['assigned_scores'].apply(lambda x: safe_json_load(x) == [0])].copy()

df['assigned_categories_scores'] = df.apply(lambda row: list(zip(safe_json_load(row['assigned_categories']), safe_json_load(row['assigned_scores']))), axis=1)
df['created_at'] = df['created_at'].astype(str).str.strip()

#Specific time of day is irrelevant for me. 
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', format='mixed', utc=True)

#Drop timezone info
df['created_at'] = df['created_at'].dt.tz_localize(None)

#Check if any have NaT in the created_at column and print their proposal_id and project_id
na_rows = df[df['created_at'].isnull()]
if not na_rows.empty:
    print("Rows with NaT in 'created_at' column:")
    print(na_rows[['proposal_id', 'project_id']])
    
df['year_month'] = df['created_at'].dt.to_period('M').astype(str)
df['year'] = df['created_at'].dt.year
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

print(f"DataFrame loaded with {len(df)} rows.")
df.head()

In [ ]:
# explode the assigned_categories_scores column, which is a list of tuples (category, score), 
# into multiple rows, and drop any rows where the score is NaN or below the confidence threshold, unless it is the only category assigned to that proposal
df_exploded = df.explode('assigned_categories_scores').reset_index(drop=True)
df_exploded[['assigned_category', 'assigned_score']] = pd.DataFrame(df_exploded['assigned_categories_scores'].tolist(), index=df_exploded.index)
df_exploded = df_exploded.drop(columns=['assigned_categories_scores'])
df_exploded = df_exploded.dropna(subset=['assigned_score'])

# Drop all rows where the assigned_category is 'Documentation & Education' if DISABLE_DOCUMENTATION_CATEGORY is True unless it is the only category assigned to that proposal
if DISABLE_DOCUMENTATION_CATEGORY:
    df_exploded['category_count'] = df_exploded.groupby('proposal_id')['assigned_category'].transform('count')
    df_exploded = df_exploded[(df_exploded['assigned_category'] != 'Documentation & Education') | (df_exploded['category_count'] == 1)]
    df_exploded = df_exploded.drop(columns=['category_count'])


df_exploded.head()

In [ ]:
# get the highest rated category for each proposal 
df_exploded['max_score'] = df_exploded.groupby('proposal_id')['assigned_score'].transform('max')

# Drop rows where the assigned_score is below the confidence threshold, or it is the highest rated category for that proposal
df_exploded = df_exploded[(df_exploded['assigned_score'] == df_exploded['max_score']) | (df_exploded['assigned_score'] >= CONFIDENCE_THRESHOLD)]

# Drop unneeded columns
df_exploded = df_exploded.drop(columns=['assigned_scores'])
df_exploded = df_exploded.drop(columns=['assigned_categories'])

print(f"Exploded DataFrame has {len(df_exploded)} rows.")
# Print count per project_id
print("Data points per project_id in the exploded dataset:")
print(df_exploded['project_id'].value_counts().sort_index())

df_exploded.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_results.csv', index=False)
df_exploded.head()

In [ ]:
json_file_path = 'categories.json'
with open(json_file_path, 'r') as f:
    categories_data = json.load(f)
    
category_to_domain = {}
for mapping in categories_data:
    for category in mapping.get('categories', []):
        category_to_domain[category] = mapping.get('domain')

In [ ]:
# Get unique from left element of tuple, else discard
all_categories = df_exploded['assigned_category'].unique()

zero_filled_frames_year_month = []
zero_filled_frames_year = []

for project, group in df_exploded.groupby('project_id'):   
    print(f"Processing project_id {project} with {len(group)} label matches after filtering...")
    
    # Determine the strict start and end dates for THIS project only
    min_date = group['created_at'].min().to_period('M')
    max_date = group['created_at'].max().to_period('M')
    
    # Generate a monthly timeline for this specific project range
    project_timeline = pd.date_range(start=min_date.to_timestamp(), end=max_date.to_timestamp(), freq='MS').to_period('M')
    
    # Create a complete cross-product grid: (All Months for Project x All 13 Categories)
    full_grid_year_month = pd.MultiIndex.from_product(
        [project_timeline, all_categories], 
        names=['year_month', 'assigned_category']
    ).to_frame(index=False)
    
    full_grid_year = pd.MultiIndex.from_product(
        [group['year'].unique(), all_categories],
        names=['year', 'assigned_category']
    ).to_frame(index=False)

    # Calculate actual proposal counts for this group
    counts_year_month = group.groupby(['year_month', 'assigned_category']).size().reset_index(name='proposal_count')
    counts_year = group.groupby(['year', 'assigned_category']).size().reset_index(name='proposal_count')

    counts_year_month['year_month'] = pd.PeriodIndex(counts_year_month['year_month'], freq='M')
    counts_year['year'] = counts_year['year'].astype(int)

    # Merge the complete month-category grid matrix with our actual data
    complete_project_df_year_month = pd.merge(full_grid_year_month, counts_year_month, on=['year_month', 'assigned_category'], how='left')
    complete_project_df_year = pd.merge(full_grid_year, counts_year, on=['year', 'assigned_category'], how='left')
    
    # Turn missing values into 0 counts
    complete_project_df_year_month['proposal_count'] = complete_project_df_year_month['proposal_count'].fillna(0).astype(int)
    complete_project_df_year_month['project_id'] = project
    complete_project_df_year_month['project_name'] = projects[project].project_name

    complete_project_df_year['proposal_count'] = complete_project_df_year['proposal_count'].fillna(0).astype(int)
    complete_project_df_year['project_id'] = project
    complete_project_df_year['project_name'] = projects[project].project_name
    
    zero_filled_frames_year_month.append(complete_project_df_year_month)
    zero_filled_frames_year.append(complete_project_df_year)

# Re-stitch all zero-filled project frameworks back together
print("\nConcatenating all project dataframes into final datasets...")
df_monthly_final = pd.concat(zero_filled_frames_year_month, ignore_index=True)
df_year_final = pd.concat(zero_filled_frames_year, ignore_index=True)

# Convert period objects back to clean strings ('YYYY-MM') for Seaborn's axis text engine
print("\nConverting 'year_month' to string format and sorting final datasets...")
df_monthly_final['year_month'] = df_monthly_final['year_month'].astype(str)
df_monthly_final = df_monthly_final.sort_values(['project_id', 'year_month']).reset_index(drop=True)

df_year_final['year'] = df_year_final['year'].astype(int)
df_year_final = df_year_final.sort_values(['project_id', 'year']).reset_index(drop=True)

# Map assigned_category to domain and create a new column 'mapped_category_domain'
df_monthly_final['mapped_category_domain'] = df_monthly_final['assigned_category'].map(category_to_domain)
df_year_final['mapped_category_domain'] = df_year_final['assigned_category'].map(category_to_domain)

# Sum up proposal counts by project_id and mapped_category_domain to get domain-level counts
df_domain_monthly = df_monthly_final.groupby(['project_id', 'mapped_category_domain', 'year_month'])['proposal_count'].sum().reset_index()
df_domain_year = df_year_final.groupby(['project_id', 'mapped_category_domain', 'year'])['proposal_count'].sum().reset_index()

df_domain_monthly['project_name'] = df_domain_monthly['project_id'].map(lambda x: projects[x].project_name)
df_domain_year['project_name'] = df_domain_year['project_id'].map(lambda x: projects[x].project_name)


print("\nSaving final datasets to CSV files...")
df_monthly_final.to_csv(f"results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_df_monthly_final.csv", index=False)
df_year_final.to_csv(f"results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_df_year_final.csv", index=False)
df_domain_monthly.to_csv(f"results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_domain_monthly.csv", index=False)
df_domain_year.to_csv(f"results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_domain_year.csv", index=False)

In [ ]:
# sum up proposal counts by project_name and mapped_category_domain to get total counts per domain for each project
df_domain_total = df_domain_year.groupby(['project_name', 'mapped_category_domain'])['proposal_count'].sum().reset_index()

# normalize for comparison
df_domain_total['proposal_count_normalized'] = df_domain_total.groupby('project_name')['proposal_count'].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)

#sort by project id
df_domain_total = df_domain_total.sort_values('project_name').reset_index(drop=True)

df_domain_total.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_domain_total.csv', index=False)



#repeat for category

df_category_total = df_year_final.groupby(['project_name', 'assigned_category'])['proposal_count'].sum().reset_index()
df_category_total['proposal_count_normalized'] = df_category_total.groupby('project_name')['proposal_count'].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)

# append an extra project called average with the average proposal count per category across all projects to df_category_total
average_category = df_category_total.groupby('assigned_category')['proposal_count'].mean().reset_index()
average_category['project_name'] = 'Average'
average_category['proposal_count_normalized'] = average_category.groupby('project_name')['proposal_count'].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)
df_category_total = pd.concat([df_category_total, average_category], ignore_index=True)


# sort by category name in the order of the category list in categories.json
with open('categories.json') as f:
    categories_data = json.load(f)
category_order = []
for mapping in categories_data:
    category_order.extend(mapping.get('categories', []))
    
df_category_total['assigned_category'] = pd.Categorical(df_category_total['assigned_category'], categories=category_order, ordered=True)

df_category_total.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_category_total.csv', index=False)



In [ ]:
# Sum up proposal counts per category per project
df_final_summed = df_year_final.copy()
df_final_summed = df_final_summed.groupby(['project_name', 'assigned_category'], as_index=False)['proposal_count'].sum()

# Calculate total proposal counts per project
df_final_summed['total_proposals'] = df_final_summed.groupby('project_name')['proposal_count'].transform('sum')

# Calculate percentage of proposals per category per project
df_final_summed['percentage'] = df_final_summed['proposal_count'] / df_final_summed['total_proposals'] * 100

# Calculate average percentage of proposals per category across all projects
df_year_summed_average = df_final_summed.copy()
df_year_summed_average['average_percentage'] = df_final_summed.groupby('assigned_category')['percentage'].transform('mean')
df_year_summed_average = df_year_summed_average.drop(columns=['project_name', 'proposal_count', 'total_proposals', 'percentage'])

df_year_summed_average.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_summed_results.csv', index=False)
df_year_summed_average.head()

In [ ]:
# Drop anything after 2025
df_year_final_summed = df_year_final[df_year_final['year'] <= 2025].copy()

# Calculate total proposal counts per project per year
df_year_final_summed['total_proposals'] = df_year_final_summed.groupby(['project_name', 'year'])['proposal_count'].transform('sum')

# Calculate percentage of proposals per category per project per year
df_year_final_summed['percentage'] = df_year_final_summed['proposal_count'] / df_year_final_summed['total_proposals'] * 100
df_year_final_summed.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_year_final_summed.csv', index=False)

# Calculate average percentage of proposals per category across all projects per year
df_year_summed_average_per_year = df_year_final_summed.copy()
df_year_summed_average_per_year['average_percentage'] = df_year_final_summed.groupby(['assigned_category', 'year'])['percentage'].transform('mean')
df_year_summed_average_per_year = df_year_summed_average_per_year.drop(columns=['project_name', 'proposal_count', 'total_proposals', 'percentage', 'mapped_category_domain', 'project_id'])

df_year_summed_average_per_year.to_csv(f'results/{DATASOURCE}_{CONFIDENCE_THRESHOLD}_summed_results_per_year.csv', index=False)
df_year_summed_average_per_year.head()